---
## Reading with values: `read_sql` and `params`

`pd.read_sql` accepts `?` placeholders just like `execute`. Pass the values as a tuple in `params`:

```python
pd.read_sql("SELECT * FROM devices WHERE active = ?", conn, params=(1,))
```

Values always go in with `?`, never an f-string. Your project's `search_records` does exactly this with `LIKE`:
`params=(f"%{text}%",)`.

In [ ]:
import sqlite3
import pandas as pd

mem = sqlite3.connect(":memory:")
pd.DataFrame({"name": ["Router-A", "Switch-B", "AP-Lobby"], "active": [1, 0, 1]}).to_sql("devices", mem, index=False)
pd.read_sql("SELECT * FROM devices WHERE active = ?", mem, params=(1,))

### Your turn
Read only the devices whose name contains `Switch`, using `LIKE ?` and `params=("%Switch%",)`.

In [ ]:
# your code here

# Week 10 · Topic 4 — Migrating Real Data In

**Box we are building: STORE, finished**

Three weeks of separate skills — Week 7's parsing, Week 8's pulled data,
Week 9's flat-file users — converging into one real database.

> **Already know this?** Skip to the *Topic 4 Challenge*.

> **Setup.** This notebook can be run on its own. The next cell connects to the same `network.db` file as Topic 1 and makes sure the `devices` table exists with two rows. It is safe to run more than once.

In [ ]:
import sqlite3

conn = sqlite3.connect("network.db")
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS devices (
        id     INTEGER PRIMARY KEY,
        name   TEXT NOT NULL,
        ip     TEXT UNIQUE NOT NULL,
        active INTEGER DEFAULT 1,
        type   TEXT
    )
""")
cursor.execute("INSERT OR IGNORE INTO devices (name, ip) VALUES ('Router-A', '10.0.0.1')")
cursor.execute("INSERT OR IGNORE INTO devices (name, ip) VALUES ('Switch-B', '10.0.0.2')")
conn.commit()
from pathlib import Path

if not Path("users.txt").exists():       # the flat file from Week 9; a sample user is created if you do not have it
    Path("users.txt").write_text("alice,$2b$12$Xhdlmin6PPDDusZa//PToOKadLYtF.6S0kKNTXRGKrkLwh8Uwsk.m\n")

---
## 1. `DataFrame.to_sql()` — the whole migration, one line

In [ ]:
import pandas as pd, requests
from io import StringIO

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df = pd.read_csv(StringIO(requests.get(url, timeout=10).text))

with sqlite3.connect("network.db") as conn:
    df.to_sql("tips", conn, if_exists="replace", index=False)
print("migrated", len(df), "rows")

### Your turn
Migrate a dataset of your own (or Week 8's fallback) into a new table with `to_sql()`.

In [ ]:
# your code here

---
## 2. `pd.read_sql()` — querying back out as a DataFrame

In [ ]:
with sqlite3.connect("network.db") as conn:
    result = pd.read_sql("SELECT * FROM tips WHERE tip > 5", conn)
print(result.head())

### Your turn
Query your migrated table back with `pd.read_sql()` using a `WHERE` condition of your own.

In [ ]:
# your code here

---
## 3. Migrating Week 9's flat file into a `users` table

Week 7's parsing pattern, applied to a security-relevant file, landing in a
real table.

In [ ]:
with sqlite3.connect("network.db") as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT NOT NULL UNIQUE,
        password_hash TEXT NOT NULL
    )""")
    with open("users.txt") as f:
        for line in f:
            username, password_hash = line.strip().split(",", 1)
            conn.execute(
                "INSERT OR IGNORE INTO users (username, password_hash) VALUES (?, ?)",
                (username, password_hash),
            )

### Your turn
Explain, in one sentence, what `INSERT OR IGNORE` does differently from a plain `INSERT` here, and why it matters if you run the migration twice.

In [ ]:
# your code here

---
## 4. Login still works exactly the same way

`checkpw()` does not care whether the hash came from a file or a database
row — Week 9's logic did not need to change at all, only where it reads from.

In [ ]:
import bcrypt

with sqlite3.connect("network.db") as conn:
    stored = conn.execute(
        "SELECT password_hash FROM users WHERE username = ?", ("alice",)
    ).fetchone()

if stored:
    print(bcrypt.checkpw(b"correct-horse-battery-staple", stored[0].encode("utf-8")))

### Your turn
Adapt your Week 9 `login_user` function so it queries the `users` table instead of reading a file.

In [ ]:
# your code here

---
## 5. Looping over the rows of a DataFrame: `iterrows()`

`to_sql()` moves a whole table at once. When each row needs a check or a special rule (as when you copy users into the `users` table
and skip any that exist already), loop over the rows with `iterrows()`. It gives the row number and the row; `row["name"]` reads one value.

`INSERT OR IGNORE` skips a row that would break a `UNIQUE` rule instead of raising an error, so the loop can be run twice safely.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (username TEXT UNIQUE, password_hash TEXT)")

old_users = pd.DataFrame({"username": ["anna", "ben"], "password_hash": ["h1", "h2"]})

for number, row in old_users.iterrows():
    conn.execute("INSERT OR IGNORE INTO users VALUES (?, ?)", (row["username"], row["password_hash"]))
conn.commit()

# running the same migration again changes nothing
for number, row in old_users.iterrows():
    conn.execute("INSERT OR IGNORE INTO users VALUES (?, ?)", (row["username"], row["password_hash"]))
conn.commit()

print(conn.execute("SELECT COUNT(*) FROM users").fetchone())

### Your turn
Add a third user to `old_users` and run both loops again. How many rows does the table have?

In [ ]:
# your code here

---
## 6. Identifying a row: `rowid`, and checking a column name: `PRAGMA`

A table made with `to_sql()` has no id column of its own, but SQLite gives every row a hidden number called `rowid`. It is a simple way to
say "this row" when you update or delete. Selecting it with a name, `rowid AS row_id`, makes it visible. Your project calls the column
`row_id`, so it cannot clash with an `id` column that already exists in your data.

A `?` placeholder protects a value, but it cannot protect a column name. Before a column name is put into an SQL string, check it against the real columns.
`PRAGMA table_info(table_name)` lists them.

In [ ]:
pd.DataFrame({"name": ["Ana", "Ben", "Cy"], "score": [7, 9, 4]}).to_sql("results", conn, index=False)

print(pd.read_sql("SELECT rowid AS row_id, * FROM results", conn))

conn.execute("UPDATE results SET score = ? WHERE rowid = ?", (10, 2))
print(conn.execute("SELECT name, score FROM results WHERE rowid = 2").fetchone())

columns = list(pd.read_sql("PRAGMA table_info(results)", conn)["name"])
print(columns)
print("score" in columns, "|", "score; DROP TABLE results" in columns)

### Your turn
Write `check_column(name)` that raises `ValueError` when `name` is not in `columns`. Test it with `"score"` and with `"oops"`.

In [ ]:
# your code here

`examples/03_rowid_and_table_info.py` shows the same ideas as a script.

---
## Topic 4 Challenge

Migrate both your project's data and your users file into one database.
Delete the old flat files once you have confirmed everything still works from
the database alone.

In [ ]:
# your code here

---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | a dataset that started as a CSV, now a queryable table |
| **Cyber Security** | incidents that started as log lines, now searchable by severity |
| **IT** | tickets that started as raw text, now filterable by status |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| running the migration twice without `OR IGNORE`/a `UNIQUE` check | `IntegrityError`, or silent duplicates | design for "may run again" from the start |
| forgetting `if_exists="replace"` (or `"append"`) on `to_sql()` | `ValueError: Table already exists` on a second run | choose deliberately, do not just guess |
| deleting the old flat file before confirming the migration worked | data loss if the migration had a bug | verify from the database first, delete the source second |
| forgetting the `.db` file needs to go in `.gitignore` | real hashed passwords committed to your repository | add it before the first commit that includes it |
| putting a column name in as a `?` | the database treats it as a text value, so nothing changes | check the name against `PRAGMA table_info`, then put it in the string |

## Go deeper

`examples/01_migrating_the_dataset.py` — to_sql, read_sql, a real dataset end to end
`examples/02_migrating_the_users_file.py` — the flat file to a table, login unchanged

---
## Recap

| You can now | Using |
|---|---|
| migrate a DataFrame in | `df.to_sql(table, conn, if_exists=...)` |
| query it back as a DataFrame | `pd.read_sql(sql, conn)` |
| migrate a flat file in | parse each line, `INSERT` |
| avoid re-migrating badly | `INSERT OR IGNORE`, or a `UNIQUE` constraint |
| process rows one at a time | `for number, row in df.iterrows():` |
| make a migration safe to repeat | `INSERT OR IGNORE` |
| refer to one row | `rowid`, shown as `rowid AS row_id` |
| list a table's columns | `PRAGMA table_info(table)` |

**You now have all of Week 10's STORE.** Open your project and migrate it for
real — nothing about it resets.